In [2]:
import pandas as pd

path = r"C:\Users\demir\retail-scorecard\data\raw\accepted_2007_to_2018Q4.csv"
sample = pd.read_csv(path, nrows=5)
sample.shape

(5, 151)

In [3]:
cols = ",\n".join(f'    "{c}" TEXT' for c in sample.columns)
print(f"CREATE TABLE raw.lc_loans (\n{cols}\n);")

CREATE TABLE raw.lc_loans (
    "id" TEXT,
    "member_id" TEXT,
    "loan_amnt" TEXT,
    "funded_amnt" TEXT,
    "funded_amnt_inv" TEXT,
    "term" TEXT,
    "int_rate" TEXT,
    "installment" TEXT,
    "grade" TEXT,
    "sub_grade" TEXT,
    "emp_title" TEXT,
    "emp_length" TEXT,
    "home_ownership" TEXT,
    "annual_inc" TEXT,
    "verification_status" TEXT,
    "issue_d" TEXT,
    "loan_status" TEXT,
    "pymnt_plan" TEXT,
    "url" TEXT,
    "desc" TEXT,
    "purpose" TEXT,
    "title" TEXT,
    "zip_code" TEXT,
    "addr_state" TEXT,
    "dti" TEXT,
    "delinq_2yrs" TEXT,
    "earliest_cr_line" TEXT,
    "fico_range_low" TEXT,
    "fico_range_high" TEXT,
    "inq_last_6mths" TEXT,
    "mths_since_last_delinq" TEXT,
    "mths_since_last_record" TEXT,
    "open_acc" TEXT,
    "pub_rec" TEXT,
    "revol_bal" TEXT,
    "revol_util" TEXT,
    "total_acc" TEXT,
    "initial_list_status" TEXT,
    "out_prncp" TEXT,
    "out_prncp_inv" TEXT,
    "total_pymnt" TEXT,
    "total_pymnt_

In [5]:
import subprocess
ddl = f"CREATE TABLE raw.lc_loans (\n{cols}\n);"
subprocess.run("clip", input=ddl, text=True)

CompletedProcess(args='clip', returncode=0)

In [7]:
import psycopg2, getpass

conn = psycopg2.connect(
    host="localhost", port=5432, dbname="retail_scorecard",
    user="postgres", password=getpass.getpass("PostgreSQL şifresi:")
)

In [8]:
with open(path, "r", encoding="utf-8") as f, conn.cursor() as cur:
    cur.copy_expert("COPY raw.lc_loans FROM STDIN WITH (FORMAT csv, HEADER true)", f)
conn.commit()

In [9]:
exprs = ",\n".join(
    f'ROUND(100.0 * COUNT("{c}") / COUNT(*))::int AS "{c}"' for c in sample.columns
)

q = f"""
SELECT RIGHT(issue_d, 4) AS issue_year,
{exprs}
FROM raw.lc_loans
WHERE TRIM(term) = '36 months'
  AND RIGHT(issue_d, 4) BETWEEN '2012' AND '2018'
GROUP BY issue_year
ORDER BY issue_year
"""

fill = pd.read_sql(q, conn).set_index("issue_year").T
fill[fill["2012"] < 95]

C:\Users\demir\AppData\Local\Temp\ipykernel_10216\134752113.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  fill = pd.read_sql(q, conn).set_index("issue_year").T


issue_year,2012,2013,2014,2015,2016,2017,2018
member_id,0,0,0,0,0,0,0
emp_title,94,93,93,93,93,92,88
desc,62,37,7,0,0,0,0
mths_since_last_delinq,41,44,51,52,53,50,45
mths_since_last_record,3,13,19,19,19,17,13
...,...,...,...,...,...,...,...
settlement_status,1,1,1,2,2,1,0
settlement_date,1,1,1,2,2,1,0
settlement_amount,1,1,1,2,2,1,0
settlement_percentage,1,1,1,2,2,1,0


In [10]:
pd.set_option("display.max_rows", 200)
fill[fill["2012"] < 95]

issue_year,2012,2013,2014,2015,2016,2017,2018
member_id,0,0,0,0,0,0,0
emp_title,94,93,93,93,93,92,88
desc,62,37,7,0,0,0,0
mths_since_last_delinq,41,44,51,52,53,50,45
mths_since_last_record,3,13,19,19,19,17,13
next_pymnt_d,0,0,0,0,28,60,88
mths_since_last_major_derog,10,20,29,30,29,27,24
annual_inc_joint,0,0,0,0,2,7,12
dti_joint,0,0,0,0,2,7,12
verification_status_joint,0,0,0,0,2,7,11
